# 00 — Real-rover validation (Week 5 artifact, Week-6/v3 refresh)

This notebook is the **human-facing wrapper** around the validation modules under `roverdevkit.validation`. All logic lives in the Python package so the checks are also enforced as CI gates in `tests/test_rover_comparison.py` and `tests/test_cross_scenario.py`.

## Goals (project_plan.md §6 W5, refreshed 2026-04-25)

The validation registry has two tiers (see `roverdevkit/validation/rover_registry.py` module docstring):

- **Flown rovers (Layer-0 truth)** — Pragyan, Yutu-2. Published mission data exists in `data/published_traverse_data.csv`, and `compare_all()` scores the evaluator against those bands.
- **Design-target rovers (inventory only)** — MoonRanger (CMU/Astrobotic, never flew), Rashid-1 (MBRSC, lost on Hakuto-R). No flight data, so they are absent from the truth table and from `compare_all()`. They exist in the registry purely so the Week-6 surrogate has design-OOD coverage on lunar micro-rover spec sheets (used by `predict_for_registry_rovers` in `roverdevkit/surrogate/baselines.py`).

The Mars-gravity Sojourner sentinel was removed in the same refactor; the project is lunar-only.

This notebook therefore:
1. Runs the end-to-end mission evaluator on the flown roster (Pragyan, Yutu-2) and compares predicted traverse, peak solar power, and thermal survival against published data.
2. Lists all four registry entries (flown + design-target) so the Week-6 sanity-check inputs are visible alongside the Week-5 acceptance gate.
3. Cross-checks evaluator behaviour across canonical scenarios (ranking tests) and single-variable perturbations (sensitivity tests).

## What counts as “validation” here
The traverse simulator produces a *design-space upper bound* on range: given the schema's floor on `drive_duty_cycle` (≥ 0.02), sim predictions are 3–8× higher than published mission totals because real missions interleave long science and thermal-wait windows that our lumped model does not capture. The Week-5 acceptance criteria (below) are phrased around *feasibility* + *thermal survival match* + *peak solar in band* rather than tight ratio-of-predicted-to-published, for exactly this reason. See the docstring of `roverdevkit.validation.rover_comparison` for the full acceptance criteria.

In [1]:
from roverdevkit.validation.rover_comparison import (
    compare_all,
    format_report,
)
from roverdevkit.validation.rover_registry import load_truth_table, registry

## 1. Registry contents

Each entry bundles a `DesignVector` (with imputation notes), a `MissionScenario` (YAML-configured), a gravity constant, a thermal architecture, and rover-specific panel parameters. Imputation notes document every field that was not directly published. The four entries cover both flown (Pragyan, Yutu-2) and design-target (MoonRanger, Rashid-1) lunar micro-rovers; only the flown two have rows in the published-truth table below and participate in `compare_all()`.

In [2]:
for entry in registry():
    print(f"--- {entry.rover_name} ---")
    print(
        f"  scenario: {entry.scenario.name}  (lat {entry.scenario.latitude_deg:+.1f} deg, {entry.scenario.mission_duration_earth_days:.0f} d, {entry.scenario.traverse_distance_m:.0f} m cap)"
    )
    print(f"  gravity: {entry.gravity_m_per_s2:.3f} m/s^2")
    print(
        f"  chassis: {entry.design.chassis_mass_kg:.1f} kg, solar: {entry.design.solar_area_m2:.2f} m^2, battery: {entry.design.battery_capacity_wh:.0f} Wh"
    )
    print(f"  panel_eff: {entry.panel_efficiency:.2f}, dust_factor: {entry.panel_dust_factor:.2f}")
    print(
        f"  thermal: A={entry.thermal_architecture.surface_area_m2:.2f} m^2, RHU={entry.thermal_architecture.rhu_power_w:.1f} W, hib={entry.thermal_architecture.hibernation_power_w:.1f} W"
    )
    print(f"  imputations: {entry.imputation_notes}")
    print()

--- Pragyan ---
  scenario: chandrayaan3_pragyan  (lat -69.4 deg, 14 d, 500 m cap)
  gravity: 1.625 m/s^2
  chassis: 10.0 kg, solar: 0.50 m^2, battery: 60 Wh
  panel_eff: 0.22, dust_factor: 0.85
  thermal: A=0.25 m^2, RHU=0.0 W, hib=2.0 W
  imputations: wheel_width, wheelbase, grouser_height/count, chassis_mass, nominal_speed_mps, drive_duty_cycle imputed from class heritage and published ops. avionics_power set to 20 W (design-space floor for a 26 kg rover).

--- Yutu-2 ---
  scenario: change4_yutu2_per_lunar_day  (lat +45.5 deg, 5 d, 200 m cap)
  gravity: 1.625 m/s^2
  chassis: 35.0 kg, solar: 1.30 m^2, battery: 130 Wh
  panel_eff: 0.20, dust_factor: 0.55
  thermal: A=0.10 m^2, RHU=15.0 W, hib=5.0 W
  imputations: chassis_mass set to 35 kg (published ex-payload chassis value; in-distribution under v3 LHS bounds 3-50 kg). Yutu-2's all-up flight mass is ~135 kg including payload, structure, and power system margins which the analytical mass-up model adds on top of chassis_mass. wheelba

## 2. Published truth table

Digitized from mission publications with explicit low/high bands. Citations live in the `citation` column of `data/published_traverse_data.csv`. The table contains only the **flown** rovers (Pragyan, Yutu-2) — design-target entries are skipped here because there is no flight data to compare against.

In [3]:
for row in load_truth_table():
    print(
        f"{row.rover_name}: traverse {row.traverse_m_published:.0f} m "
        f"(band {row.traverse_m_low:.0f}-{row.traverse_m_high:.0f}), "
        f"peak solar {row.peak_solar_power_w_published:.0f} W "
        f"(band {row.peak_solar_power_w_low:.0f}-{row.peak_solar_power_w_high:.0f}), "
        f"thermal survival = {row.thermal_survival_published}"
    )
    print(f"  citation: {row.citation}")
    print(f"  note: {row.notes}")
    print()

Pragyan: traverse 101 m (band 80-140), peak solar 50 W (band 40-70), thermal survival = False
  citation: ISRO Chandrayaan-3 mission updates (Aug-Sep 2023); Nature SR 14:24178 (2024)
  note: Traverse over Lunar Day 1 only. Rover did NOT survive lunar night (no RHUs); thermal_survival_published=false matches the sim's full-mission hot+cold steady-state check. traverse_m_published is the in-mission total.

Yutu-2: traverse 25 m (band 10-60), peak solar 135 W (band 110-160), thermal survival = True
  citation: Di et al. 2020 Icarus; Ding et al. 2022 Acta Astronautica; CNSA dispatches
  note: Per-lunar-day drive distance, first ~2 years. Yutu-2 carries Pu-238 RHUs for lunar-night survival (surviving 60+ lunar days as of 2025); thermal_survival_published=true conditional on registry's RHU-carrying architecture.



## 3. Run the evaluator on every flown rover and score

`compare_all` iterates over `flown_registry()` (Pragyan, Yutu-2), runs `evaluate()` on each entry, and applies the Week-5 acceptance criteria. The resulting `ComparisonSummary` is what `tests/test_rover_comparison.py` asserts on.

In [4]:
summary = compare_all()
print(format_report(summary))

Rover      range_pred   range_pub  range_ratio  peak_solar_pred  peak_solar_pub  thermal  motor  PASS?
---------------------------------------------------------------------------------------------------------
Pragyan         500 m       101 m       4.93x          44.8 W          50.0 W  match   ok     PASS
Yutu-2          200 m        25 m       8.00x         136.4 W         135.0 W  match   ok     PASS
---------------------------------------------------------------------------------------------------------
Pass rate: 2/2


## 4. Per-rover acceptance-criteria breakdown

A more verbose view: each criterion, one line per rover.

In [5]:
for r in summary.results:
    print(f"--- {r.rover_name} --- {'PASS' if r.passes else 'FAIL'}")
    print(
        f"  range_feasible          = {r.range_feasible}  (predicted {r.range_m_predicted:.0f} m vs low bound {r.truth.traverse_m_low:.0f} m)"
    )
    print(f"  range_below_sanity_ceil = {r.range_below_sanity_ceiling}")
    print(
        f"  thermal_matches         = {r.thermal_matches}  (predicted {r.metrics.thermal_survival}, published {r.truth.thermal_survival_published})"
    )
    print(f"  motor_and_traversal_ok  = {r.motor_and_traversal_ok}")
    print(
        f"  peak_solar_in_band      = {r.peak_solar_in_band}  (predicted {r.peak_solar_power_w_predicted:.1f} W, band {r.truth.peak_solar_power_w_low:.0f}-{r.truth.peak_solar_power_w_high:.0f} W)"
    )
    print()

--- Pragyan --- PASS
  range_feasible          = True  (predicted 500 m vs low bound 80 m)
  range_below_sanity_ceil = True
  thermal_matches         = True  (predicted False, published False)
  motor_and_traversal_ok  = True
  peak_solar_in_band      = True  (predicted 44.8 W, band 40-70 W)

--- Yutu-2 --- PASS
  range_feasible          = True  (predicted 200 m vs low bound 10 m)
  range_below_sanity_ceil = True
  thermal_matches         = True  (predicted True, published True)
  motor_and_traversal_ok  = True
  peak_solar_in_band      = True  (predicted 136.4 W, band 110-160 W)



## 5. Cross-scenario ranking

Three hand-crafted design archetypes (`large_traverser`, `polar_survivor`, `slope_climber`) evaluated on each canonical scenario. We check that the slope-specialty archetype wins slope capability, and that range is dominated by the fast archetype in benign terrain.

In [6]:
from roverdevkit.validation.cross_scenario import rank_archetypes

for scen_name, rank in rank_archetypes().items():
    print(f"--- {scen_name} ---")
    print(f"  range winner:             {rank.range_winner}")
    print(f"  slope capability winner:  {rank.slope_capability_winner}")
    print(f"  energy margin winner:     {rank.energy_margin_winner}")
    for name, m in rank.per_archetype.items():
        print(
            f"    {name:16s}: range={m.range_km * 1000:6.0f} m  slope_cap={m.slope_capability_deg:4.1f} deg  margin={m.energy_margin_pct:5.1f}%  mass={m.total_mass_kg:5.1f} kg"
        )
    print()

--- equatorial_mare_traverse ---
  range winner:             large_traverser
  slope capability winner:  slope_climber
  energy margin winner:     large_traverser
    large_traverser : range= 48384 m  slope_cap=20.9 deg  margin=100.0%  mass= 46.1 kg
    polar_survivor  : range=  4838 m  slope_cap=16.2 deg  margin=100.0%  mass= 22.8 kg
    slope_climber   : range= 14515 m  slope_cap=22.9 deg  margin=100.0%  mass= 54.1 kg

--- polar_prospecting ---
  range winner:             large_traverser
  slope capability winner:  slope_climber
  energy margin winner:     large_traverser
    large_traverser : range= 30000 m  slope_cap=20.9 deg  margin=  0.0%  mass= 46.1 kg
    polar_survivor  : range= 10368 m  slope_cap=16.2 deg  margin=  0.0%  mass= 22.8 kg
    slope_climber   : range= 30000 m  slope_cap=22.9 deg  margin=  0.0%  mass= 54.1 kg

--- highland_slope_capability ---
  range winner:             large_traverser
  slope capability winner:  slope_climber
  energy margin winner:     large_tra

## 6. One-at-a-time sensitivity

Bump each design variable in turn around a baseline (`_baseline_design()` in `roverdevkit/validation/cross_scenario.py`) and look at the sign of the change in each mission metric. The CI tests enforce the expected direction.

**How to read the table.** Each row is a separate simulation where one variable is changed from `baseline` to `bumped` and everything else is held fixed. The `d_*` columns are absolute deltas, `metric(bumped) - metric(baseline)`:

- `d_range_km` — change in traverse range (km)
- `d_margin_pct` — change in SOC-based energy margin (percentage points, reporting metric; clipped at 0 % / 100 %)
- `d_raw_pct` — change in **integrated-energy margin** `(E_gen - E_used) / E_used * 100` (unbounded, surrogate-training metric)
- `d_slope_deg` — change in slope capability (degrees)
- `d_mass_kg` — change in total mass (kg)

The scenario is distance-unlimited (`_sensitivity_scenario`: synthetic 100 km equatorial mare, 14 days) so range is energy/duty-cycle bound rather than clipped at `traverse_distance_m`. The baseline is deliberately slow and low-duty for the same reason.

**Why `d_margin_pct` reads `+0.00` for most rows.** The baseline is energy-positive on this scenario so the SOC-based margin clips at 100 %; adding more solar area or battery capacity doesn't move the reporting metric. `d_raw_pct` is the unclipped companion that remains sensitive across the whole design space — it's what the Phase-2 surrogate will actually be trained on. Every row produces a non-zero `d_raw_pct` with the physically expected sign; the CI tests (`test_cross_scenario.py`) enforce the key signs (solar → positive, avionics → negative) strictly.

In [7]:
from roverdevkit.validation.cross_scenario import one_at_a_time_sensitivity

header = (
    f"{'variable':22s} {'baseline':>10s} {'bumped':>10s}  "
    f"{'d_range_km':>12s} {'d_margin_pct':>14s} {'d_raw_pct':>12s} "
    f"{'d_slope_deg':>12s} {'d_mass_kg':>10s}"
)
print(header)
print("-" * len(header))
for entry in one_at_a_time_sensitivity():
    print(
        f"{entry.variable:22s} {entry.baseline_value:>10.3f} {entry.bumped_value:>10.3f}  "
        f"{entry.delta_range_km:+12.3f} {entry.delta_energy_margin_pct:+14.2f} "
        f"{entry.delta_energy_margin_raw_pct:+12.2f} "
        f"{entry.delta_slope_capability_deg:+12.2f} {entry.delta_total_mass_kg:+10.3f}"
    )

variable                 baseline     bumped    d_range_km   d_margin_pct    d_raw_pct  d_slope_deg  d_mass_kg
--------------------------------------------------------------------------------------------------------------


solar_area_m2               0.400      0.700        +0.000          +0.00      +427.93        +0.05     +1.060
battery_capacity_wh       100.000    200.000        +0.000          +0.00        -0.12        +0.06     +1.178
avionics_power_w           15.000     25.000        +0.000         -12.43      -227.76        +0.04     +0.707
chassis_mass_kg            12.000     20.000        +0.000          +0.00        -1.22        +0.45    +11.306
wheel_radius_m              0.120      0.160        +0.000          +0.00        +0.25        +3.25     +1.730
nominal_speed_mps           0.020      0.050        +5.443          +0.00        -3.73        +0.00     +0.000
drive_duty_cycle            0.150      0.350        +4.838          +0.00        -3.32        +0.00     +0.000


## 7. Known limitations of the Week-5 harness

Documented honestly so these are **in-plan** rather than undiscovered blindspots:

1. **Range is a capability-envelope prediction, not an operational prediction.** `MissionMetrics.range_km` is the distance the hardware can deliver *if ops commands it to drive at its designed* `drive_duty_cycle` *throughout the mission window*. Real missions typically command a fraction of the designed duty (Pragyan ~0.02, Yutu-2 ~0.015) for commanding / thermal-window / science-campaign reasons the evaluator doesn't model. Predicted envelope range is 3–8× published *actual* range for this structural reason, which is expected and correct. Post-hoc utilisation is handled by `evaluator.range_at_utilisation(metrics, design, u)` rather than by re-running the sim. The Week 5 acceptance gate verifies envelope feasibility (`predicted ≥ published floor`) rather than tight ratios.

2. **Analytical terramechanics floor.** The Bekker-Wong path underestimates DP on loose soil (∼20–40 %) relative to what Ding 2011 testbeds show. Week 7 will add an SCM correction to close this gap; Week 5 acceptance criteria are set loose enough to tolerate the current under-prediction.

3. **Yutu-2 sits at the heavy end of the v3 design space.** Yutu-2 has a published all-up mass of ~135 kg; the surrogate's design space targets lunar micro-rovers (chassis 3–50 kg ex-payload, total ≤ ~100 kg with payload). Yutu-2's `chassis_mass_kg` is held at 35 kg ex-payload (the literature value for the chassis line item) rather than the all-up flight mass, which is the right thing to feed an analytical mass-up model. After the v3 LHS bounds widening (chassis to 50 kg, wheel widths to 0.20 m) it sits inside the surrogate's training support rather than at a corner.

4. **Thermal model is single-node.** Real rovers use MLI + variable-emittance louvers + heat pipes. We represent MLI via effective-radiating-area and low absorptivity. Yutu-2's RHU power is tuned to a plausible effective value rather than the actual RHU wattage.

5. **Constant-slope traverse.** `traverse_sim.py` treats `scenario.max_slope_deg` as a fixed slope for the whole run. We configured the validation scenarios at typical-ops slope (5–6°), not worst-case (12–15°), to match how the real rovers drove. A proper terrain-path model is v2 work.

6. **Panel and thermal parameters are per-rover calibrated, not independently validated.** `RoverRegistryEntry` carries rover-specific `panel_efficiency`, `panel_dust_factor`, and a `thermal_architecture` block (effective radiating area, absorptivity, RHU wattage, operating-temp ceiling). These are set to vendor-datasheet values for each rover's flight panel (Spectrolab triple-junction for Pragyan, solar-cell-of-the-era for Yutu-2) and to effective-parameter MLI/RHU settings that match the published thermal outcome. This is honest calibration of an effective-parameter model, not independent validation of the solar-geometry or Stefan-Boltzmann math underneath. The Week-9 error-budget deliverable should separate these two layers explicitly.

7. **Non-binding `traverse_distance_m` on canonical scenarios.** The four canonical scenario YAMLs were Week-5-originally short-mission budgets (0.5–5 km) representing specific assigned traverses. After the Week-5 review we raised them to the 14–30-day theoretical reach so that the Phase-2 LHS sweep sees a range signal that responds to speed/duty/energy levers instead of saturating at a distance cap. The flown-rover validation scenarios (`chandrayaan3_pragyan`, `change4_yutu2_per_lunar_day`) keep their short-mission caps because they're being compared against a specific published traverse. The two design-target scenarios (`moonranger_polar_demo`, `rashid_atlas_crater`) carry their published mission-design distance caps for the same reason.